# Wave Equation and C Code Generation

## Authors: Zach Etienne and Thiago Assumpcao

To run the whole notebook, click the `>>` toolbar button and choose
**Restart Kernel and Run All Cells...**.

This notebook builds the Cartesian scalar-wave right-hand sides, checks them
against the hand-written first-order system, and emits complete C assignment
blocks that later appear inside generated wave projects.

**Notebook Status:** Validated executable tutorial.

**Validation Notes:** The validation section compares the NRPy symbolic
right-hand sides with the hand-written Cartesian formulas and raises an error
unless both residuals vanish.

Navigation: [Index](../index.ipynb) |
Previous: [Finite-Difference Playground][fd-playground] |
Next: [Start-to-Finish Cartesian Wave Project](start_to_finish_wave_cartesian.ipynb)

[fd-playground]: ../2-numerical_methods/finite_difference_playground.ipynb

# Table of Contents

1. [Required Reading and Source Code](#Required-Reading-and-Source-Code)
1. [Words for This Notebook](#Words-for-This-Notebook)
1. [Initial-Value Problem](#Initial-Value-Problem)
1. [Step 1: Import the Symbolic and Codegen Tools](#Step-1:-Import-the-Symbolic-and-Codegen-Tools)
1. [Step 2](#Step-2:-Build-the-Cartesian-Wave-Right-Hand-Sides)
1. [Validation Check](#Validation-Check)
1. [Step 3](#Step-3:-Compare-Direct-and-CSE-Generated-C-Assignments)
1. [Step 4: Emit Exact-Solution Assignments](#Step-4:-Emit-Exact-Solution-Assignments)
1. [What next?](#What-next?)

# Required Reading and Source Code
### [Back to [top](#Table-of-Contents)]

This notebook assumes the learner has seen finite-difference notation, the
Method of Lines, and basic C assignment generation:

- [Finite-Difference Playground](../2-numerical_methods/finite_difference_playground.ipynb)
- [C Code Generation](../1-intro/c_codegen.ipynb)
- [Method of Lines and RK](../2-numerical_methods/method_of_lines_and_rk.ipynb)

Installed NRPy modules used here:

- `nrpy.equations.wave_equation.WaveEquation_RHSs`
  (`equations/wave_equation/WaveEquation_RHSs.py`): builds symbolic
  Cartesian wave right-hand sides.
- `nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData`
  (`equations/wave_equation/WaveEquation_Solutions_InitialData.py`):
  provides analytic wave solutions for initial data and diagnostics.
- `nrpy.c_codegen` (`c_codegen.py`): converts SymPy expressions into
  C assignments.

These are package-relative paths inside the pip-installed `nrpy`; no cloned
source checkout is required.

# Words for This Notebook
### [Back to [top](#Table-of-Contents)]

- **`u`:** the scalar wave amplitude.
- **`v`:** the auxiliary field `v = partial_t u`, introduced so the wave
  equation is first order in time.
- **Wave speed:** the parameter `wavespeed = c` that sets how fast the wave
  propagates in the chosen units.
- **First-order system:** a rewrite where each evolution equation contains
  only first time derivatives.
- **Right-hand side:** the expression after the equals sign in an evolution
  equation, such as `partial_t u = v`.
- **`uu_rhs`, `vv_rhs`:** NRPy symbolic names for the right-hand sides of
  `u` and `v`.
- **Exact solution:** a known analytic solution used as trusted data.
- **Residual:** the simplified difference between two formulas; zero means the
  formulas match.
- **C assignment:** a generated C statement that stores one expression in one
  named variable.
- **Common subexpression elimination (CSE):** a code-generation optimization
  that stores a repeated expression in a temporary variable such as `tmp0`.
- **BHaH:** the standalone NRPy code-writing infrastructure selected here so
  generated names match the project notebooks.

# Initial-Value Problem
### [Back to [top](#Table-of-Contents)]

In Cartesian coordinates `(x, y, z)`, the scalar wave equation is

$$
\partial_t^2 u = c^2
\left(\partial_x^2 u + \partial_y^2 u + \partial_z^2 u\right).
$$

We introduce `v = partial_t u` and evolve the first-order-in-time system

$$
\partial_t u = v, \qquad
\partial_t v = c^2
\left(\partial_x^2 u + \partial_y^2 u + \partial_z^2 u\right).
$$

The domain is a Cartesian grid in code-generation notation. This notebook does
not evolve the grid or impose boundary conditions; later project notebooks add
finite differences, ghost zones, Method-of-Lines time stepping, and boundary
handling. The trusted analytic data used here are plane-wave expressions from
NRPy. The expected behavior is exact symbolic agreement between the NRPy
right-hand sides and the hand-written formulas.

# Step 1: Import the Symbolic and Codegen Tools
### [Back to [top](#Table-of-Contents)]

This setup cell imports SymPy and the NRPy modules used below. It has no
lesson output; the next cells print the symbolic expressions and validation
residuals.

In [1]:
import sympy as sp

import nrpy.c_codegen as ccg
import nrpy.grid as gri
import nrpy.indexedexp as ixp
import nrpy.params as par
from nrpy.equations.wave_equation.WaveEquation_RHSs import WaveEquation_RHSs
from nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData import (
    WaveEquation_solution_Cartesian,
)

# Step 2: Build the Cartesian Wave Right-Hand Sides
### [Back to [top](#Table-of-Contents)]

The mathematical objects are the two right-hand sides in the first-order wave
system. NRPy stores them as SymPy expressions in `rhs.uu_rhs` and `rhs.vv_rhs`.
Inspect:

- `uu_rhs`, which should be the auxiliary field `v`;
- `vv_rhs`, which should be `wavespeed**2` times the three Cartesian second
  derivatives of `u`.

In [2]:
for name in ["uu", "vv", "uu_rhs", "vv_rhs"]:
    gri.glb_gridfcs_dict.pop(name, None)
par.set_parval_from_str("Infrastructure", "BHaH")
rhs = WaveEquation_RHSs()
print("uu_rhs =", rhs.uu_rhs)
print("vv_rhs =", rhs.vv_rhs)

uu_rhs = vv
vv_rhs = uu_dDD00*wavespeed**2 + uu_dDD11*wavespeed**2 + uu_dDD22*wavespeed**2


# Validation Check
### [Back to [top](#Table-of-Contents)]

The trusted result is the hand-written first-order Cartesian wave system above.
The newly computed result is the NRPy symbolic expression. The check passes
only if both simplified residuals are exactly zero.

In [3]:
wavespeed = sp.Symbol("wavespeed", real=True)
vv = sp.Symbol("vv", real=True)
uu_dDD = ixp.declarerank2("uu_dDD", symmetry="sym01")
hand_uu_rhs = vv
hand_vv_rhs = wavespeed**2 * sum(uu_dDD[i][i] for i in range(3))
uu_residual = sp.simplify(rhs.uu_rhs - hand_uu_rhs)
vv_residual = sp.simplify(rhs.vv_rhs - hand_vv_rhs)
print("uu residual:", uu_residual)
print("vv residual:", vv_residual)
if uu_residual != 0 or vv_residual != 0:
    raise RuntimeError("Expected the NRPy wave right-hand sides to match.")
print("PASS: NRPy right-hand sides match the hand-written Cartesian formulas.")

uu residual: 0
vv residual: 0
PASS: NRPy right-hand sides match the hand-written Cartesian formulas.


# Step 3: Compare Direct and CSE-Generated C Assignments
### [Back to [top](#Table-of-Contents)]

The next outputs are complete C assignment blocks for the two evolution
right-hand sides. Inspect:

- the direct output, where `wavespeed*wavespeed` appears wherever needed;
- the CSE output, where the repeated square is stored once in `tmp0`;
- the final assignments to `uu_rhs` and `vv_rhs`.

In [4]:
rhs_expressions = [rhs.uu_rhs, rhs.vv_rhs]
rhs_names = ["uu_rhs", "vv_rhs"]
direct_assignments = ccg.c_codegen(
    rhs_expressions,
    rhs_names,
    enable_cse=False,
    include_braces=False,
    verbose=False,
)
cse_assignments = ccg.c_codegen(
    rhs_expressions,
    rhs_names,
    enable_cse=True,
    include_braces=False,
    verbose=False,
)
print("complete direct right-hand-side assignments:")
print(direct_assignments)
print("complete CSE right-hand-side assignments:")
print(cse_assignments)

complete direct right-hand-side assignments:
uu_rhs = vv;
vv_rhs = uu_dDD00*((wavespeed)*(wavespeed)) + uu_dDD11*((wavespeed)*(wavespeed)) + uu_dDD22*((wavespeed)*(wavespeed));

complete CSE right-hand-side assignments:
const REAL tmp0 = ((wavespeed)*(wavespeed));
uu_rhs = vv;
vv_rhs = tmp0*uu_dDD00 + tmp0*uu_dDD11 + tmp0*uu_dDD22;



The direct and CSE blocks compute the same formulas. CSE changes only the code
shape: `tmp0` stores `wavespeed**2`, and the `vv_rhs` assignment reuses that
temporary for all three second-derivative terms.

# Step 4: Emit Exact-Solution Assignments
### [Back to [top](#Table-of-Contents)]

The trusted plane-wave solution supplies analytic `u` and `v` values. Complete
projects use the same expressions to set initial data and to compute diagnostic
relative errors against known values.

In [5]:
plane_wave = WaveEquation_solution_Cartesian(WaveType="PlaneWave")
exact_solution_assignments = ccg.c_codegen(
    [plane_wave.uu_exactsoln, plane_wave.vv_exactsoln],
    ["uu_exact", "vv_exact"],
    include_braces=False,
    verbose=False,
)
print("complete exact-solution assignments:")
print(exact_solution_assignments)

complete exact-solution assignments:
const REAL tmp0 = time*wavespeed - (kk0*xCart0 + kk1*xCart1 + kk2*xCart2)/sqrt(((kk0)*(kk0)) + ((kk1)*(kk1)) + ((kk2)*(kk2)));
uu_exact = 2 - sin(tmp0);
vv_exact = -wavespeed*cos(tmp0);



The residuals prove that the symbolic wave right-hand sides match the hand
formulas. The generated C blocks show how those expressions become assignment
bodies for generated projects, and the exact-solution block provides trusted
analytic values for initialization and diagnostics.

# What next?
### [Back to [top](#Table-of-Contents)]

- [Start-to-Finish Cartesian Wave Project](start_to_finish_wave_cartesian.ipynb)
- [C Code Generation](../1-intro/c_codegen.ipynb)
- [Finite Differences](../1-intro/finite_difference.ipynb)